# Cycle & Sleep: Hormonal Patterns in Sleep Quality

Investigamos si el ciclo menstrual influye en la duración, estructura y calidad del sueño.

**Datos**: Apple Health export (~6 años)  
**Métricas de sueño**: duración total, % REM, % Deep, eficiencia, despertares  
**Métricas del ciclo**: fase (menstrual, folicular, ovulatoria, lútea), temperatura de muñeca

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import timedelta
from pathlib import Path

from redmoon.constants import (
    PHASE_ORDER, PHASE_COLORS, METRIC_LABELS,
    MIN_SLEEP_MIN, MAX_INBED_MIN, EARLY_MORNING_CUTOFF,
    MIN_CYCLE_DAYS, MAX_CYCLE_DAYS, NEW_PERIOD_GAP_DAYS,
    assign_phase,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

# Use personal data if available, otherwise sample data
_root = Path("..") 
DATA_DIR = str(_root / "data") if (_root / "data" / "sleep.csv").exists() else str(_root / "sample_data")
print(f"Using data from: {DATA_DIR}")


## 1. Cargar datos

In [ ]:
sleep_raw = pd.read_csv(f"{DATA_DIR}/sleep.csv", parse_dates=["start", "end"])
menstrual = pd.read_csv(f"{DATA_DIR}/menstrual.csv", parse_dates=["date"])
wrist_temp = pd.read_csv(f"{DATA_DIR}/wrist_temp.csv", parse_dates=["date"])
breathing = pd.read_csv(f"{DATA_DIR}/breathing.csv", parse_dates=["date"])
hrv = pd.read_csv(f"{DATA_DIR}/hrv.csv", parse_dates=["datetime"])
resting_hr = pd.read_csv(f"{DATA_DIR}/resting_hr.csv", parse_dates=["date"])

print(f"Sleep records: {len(sleep_raw):,}")
print(f"Menstrual records: {len(menstrual):,}")
print(f"Wrist temp records: {len(wrist_temp):,}")
print(f"Breathing records: {len(breathing):,}")
print(f"HRV records: {len(hrv):,}")
print(f"Resting HR records: {len(resting_hr):,}")
print(f"\nSleep stages: {sleep_raw['stage'].value_counts().to_dict()}")
print(f"Flow types: {menstrual['flow'].value_counts().to_dict()}")
print(f"\nSleep date range: {sleep_raw['start'].min().date()} → {sleep_raw['start'].max().date()}")
print(f"Menstrual date range: {menstrual['date'].min().date()} → {menstrual['date'].max().date()}")

## 2. Agregar sueño por noche

Cada noche se asigna a la fecha en que empieza el sueño (si empiezas a dormir a las 23:00 del día 5, esa noche cuenta como día 5). Para sueño que empieza después de medianoche pero antes de las 6:00, se asigna al día anterior.

In [ ]:
# Assign each sleep record to a "sleep night" date
sleep = sleep_raw.copy()
sleep["hour"] = sleep["start"].dt.hour
# If sleep starts between midnight and 6am, assign to previous day
sleep["night_date"] = sleep["start"].dt.date
mask_early = sleep["hour"] < 6
sleep.loc[mask_early, "night_date"] = (sleep.loc[mask_early, "start"] - timedelta(days=1)).dt.date
sleep["night_date"] = pd.to_datetime(sleep["night_date"])

# Aggregate per night
stages_of_interest = ["AsleepCore", "AsleepREM", "AsleepDeep", "Awake", "InBed"]

nightly = []
for night, group in sleep.groupby("night_date"):
    row = {"night_date": night}
    
    for stage in stages_of_interest:
        stage_data = group[group["stage"] == stage]
        row[f"{stage}_min"] = stage_data["duration_min"].sum()
    
    # Total sleep = Core + REM + Deep (+ Unspecified if present)
    unspecified = group[group["stage"] == "AsleepUnspecified"]["duration_min"].sum()
    row["total_sleep_min"] = row["AsleepCore_min"] + row["AsleepREM_min"] + row["AsleepDeep_min"] + unspecified
    
    # Total in-bed: use InBed if available, otherwise sum all stages
    total_all = row["total_sleep_min"] + row["Awake_min"]
    row["total_inbed_min"] = row["InBed_min"] if row["InBed_min"] >= total_all else total_all
    
    # Percentages
    if row["total_sleep_min"] > 0:
        row["pct_rem"] = row["AsleepREM_min"] / row["total_sleep_min"] * 100
        row["pct_deep"] = row["AsleepDeep_min"] / row["total_sleep_min"] * 100
        row["pct_core"] = row["AsleepCore_min"] / row["total_sleep_min"] * 100
    else:
        row["pct_rem"] = row["pct_deep"] = row["pct_core"] = np.nan
    
    # Efficiency: sleep time / in-bed time (cap at 100%)
    if row["total_inbed_min"] > 0:
        row["efficiency"] = min(row["total_sleep_min"] / row["total_inbed_min"] * 100, 100.0)
    else:
        row["efficiency"] = np.nan
    
    # Number of awakenings
    row["n_awakenings"] = len(group[group["stage"] == "Awake"])
    
    # Sleep start/end times
    row["sleep_start"] = group["start"].min()
    row["sleep_end"] = group["end"].max()
    
    nightly.append(row)

nightly_df = pd.DataFrame(nightly)

# Filter: only nights with meaningful sleep data (>2h sleep, <16h in bed)
nightly_df = nightly_df[
    (nightly_df["total_sleep_min"] > 120) &
    (nightly_df["total_inbed_min"] < 960)
].reset_index(drop=True)

print(f"Noches válidas: {len(nightly_df)}")
print(f"Eficiencia media: {nightly_df['efficiency'].mean():.1f}% (rango: {nightly_df['efficiency'].min():.1f}% - {nightly_df['efficiency'].max():.1f}%)")
nightly_df.head()

## 3. Identificar ciclos menstruales y asignar fases

Agrupamos los días de sangrado consecutivos en periodos, y estimamos las fases del ciclo entre cada periodo.

In [ ]:
# Identify period start dates (first day of bleeding after a gap of >5 days)
menstrual_sorted = menstrual.sort_values("date").reset_index(drop=True)
menstrual_sorted["date_dt"] = pd.to_datetime(menstrual_sorted["date"])
menstrual_sorted["gap"] = menstrual_sorted["date_dt"].diff().dt.days

# A new period starts when gap > 5 days (or first record)
menstrual_sorted["new_period"] = (menstrual_sorted["gap"] > 5) | (menstrual_sorted["gap"].isna())
menstrual_sorted["period_id"] = menstrual_sorted["new_period"].cumsum()

# Get period start dates and durations
periods = menstrual_sorted.groupby("period_id").agg(
    start=('date_dt', 'min'),
    end=('date_dt', 'max'),
    n_days=('date_dt', 'count')
).reset_index()

# Cycle length = days between period starts
periods["cycle_length"] = periods["start"].diff().dt.days

print(f"Periodos identificados: {len(periods)}")
print(f"Duración media del ciclo: {periods['cycle_length'].mean():.1f} días (σ={periods['cycle_length'].std():.1f})")
print(f"Duración media del sangrado: {periods['n_days'].mean():.1f} días")
print()
periods.tail(10)

In [ ]:
# Visualize cycle length over time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
valid_cycles = periods.dropna(subset=["cycle_length"])
ax.plot(valid_cycles["start"], valid_cycles["cycle_length"], "o-", markersize=4, alpha=0.7)
ax.axhline(y=28, color="red", linestyle="--", alpha=0.5, label="28 días")
ax.set_title("Duración del ciclo a lo largo del tiempo")
ax.set_ylabel("Días")
ax.legend()

ax = axes[1]
ax.hist(valid_cycles["cycle_length"], bins=20, edgecolor="white", alpha=0.8)
ax.axvline(x=valid_cycles["cycle_length"].median(), color="red", linestyle="--", label=f"Mediana: {valid_cycles['cycle_length'].median():.0f}d")
ax.set_title("Distribución de la duración del ciclo")
ax.set_xlabel("Días")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Use shared phase-assignment from redmoon.constants
# (imported in cell 1: assign_phase, PHASE_ORDER, MIN/MAX_CYCLE_DAYS)

phases = nightly_df["night_date"].apply(lambda d: assign_phase(d, periods))
nightly_df["phase"] = phases.apply(lambda x: x[0])
nightly_df["cycle_day"] = phases.apply(lambda x: x[1])
nightly_df["cycle_length"] = phases.apply(lambda x: x[2])

# Filter to only nights within a known cycle
cycle_sleep = nightly_df.dropna(subset=["phase"]).copy()
cycle_sleep = cycle_sleep[
    (cycle_sleep["cycle_length"] >= MIN_CYCLE_DAYS) & (cycle_sleep["cycle_length"] <= MAX_CYCLE_DAYS)
]

print(f"Noches con fase del ciclo asignada: {len(cycle_sleep)}")
print(f"\nDistribucion por fase:")
print(cycle_sleep["phase"].value_counts())


## 4. Merge con temperatura y respiración

In [ ]:
# Merge wrist temperature
wrist_temp["night_date"] = pd.to_datetime(wrist_temp["date"])
cycle_sleep = cycle_sleep.merge(wrist_temp[["night_date", "temp_c"]], on="night_date", how="left")

# Merge breathing disturbances
breathing["night_date"] = pd.to_datetime(breathing["date"])
cycle_sleep = cycle_sleep.merge(breathing[["night_date", "disturbances"]], on="night_date", how="left")

# Merge daily HRV (mean per day)
hrv["date"] = pd.to_datetime(hrv["datetime"]).dt.date
hrv_daily = hrv.groupby("date")["hrv_ms"].mean().reset_index()
hrv_daily["night_date"] = pd.to_datetime(hrv_daily["date"])
cycle_sleep = cycle_sleep.merge(hrv_daily[["night_date", "hrv_ms"]], on="night_date", how="left")

# Merge resting heart rate
resting_hr["night_date"] = pd.to_datetime(resting_hr["date"])
cycle_sleep = cycle_sleep.merge(resting_hr[["night_date", "resting_hr_bpm"]], on="night_date", how="left")

print(f"Noches con temperatura: {cycle_sleep['temp_c'].notna().sum()}")
print(f"Noches con respiración: {cycle_sleep['disturbances'].notna().sum()}")
print(f"Noches con HRV: {cycle_sleep['hrv_ms'].notna().sum()}")
print(f"Noches con resting HR: {cycle_sleep['resting_hr_bpm'].notna().sum()}")

## 5. Análisis descriptivo por fase del ciclo

In [ ]:
# phase_order imported from redmoon.constants (PHASE_ORDER)
metrics = ["total_sleep_min", "pct_rem", "pct_deep", "pct_core", "efficiency", "n_awakenings"]
metric_labels = {
    "total_sleep_min": "Duración total (min)",
    "pct_rem": "% REM",
    "pct_deep": "% Deep",
    "pct_core": "% Core",
    "efficiency": "Eficiencia (%)",
    "n_awakenings": "Despertares"
}

summary = cycle_sleep.groupby("phase")[metrics].agg(["mean", "std", "median", "count"]).round(2)
summary = summary.reindex(PHASE_ORDER)
summary

In [ ]:
# Box plots for each metric by cycle phase
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = {"Menstrual": "#e74c3c", "Folicular": "#3498db", "Ovulatoria": "#2ecc71", "Lútea": "#f39c12"}

for ax, metric in zip(axes.flat, metrics):
    data = [cycle_sleep[cycle_sleep["phase"] == p][metric].dropna() for p in phase_order]
    bp = ax.boxplot(data, tick_labels=phase_order, patch_artist=True, widths=0.6)
    for patch, phase in zip(bp["boxes"], phase_order):
        patch.set_facecolor(colors[phase])
        patch.set_alpha(0.6)
    ax.set_title(metric_labels[metric], fontsize=13, fontweight="bold")
    ax.tick_params(axis="x", rotation=15)

fig.suptitle("Métricas de sueño por fase del ciclo menstrual", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6. Tests estadísticos

Kruskal-Wallis test (no-paramétrico) para cada métrica entre las 4 fases, seguido de tests post-hoc de Dunn si hay significancia.

In [ ]:
from itertools import combinations

print("=" * 70)
print("KRUSKAL-WALLIS TEST: Diferencias entre fases del ciclo")
print("=" * 70)

significant_metrics = []

for metric in metrics:
    groups = [cycle_sleep[cycle_sleep["phase"] == p][metric].dropna() for p in PHASE_ORDER]
    stat, p_val = stats.kruskal(*groups)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
    print(f"\n{metric_labels[metric]}:")
    print(f"  H = {stat:.3f}, p = {p_val:.6f} {sig}")
    
    if p_val < 0.05:
        significant_metrics.append(metric)
        # Post-hoc: pairwise Mann-Whitney U tests with Bonferroni correction
        n_comparisons = 6  # C(4,2)
        for (i, p1), (j, p2) in combinations(enumerate(PHASE_ORDER), 2):
            u_stat, u_p = stats.mannwhitneyu(groups[i], groups[j], alternative="two-sided")
            adj_p = min(u_p * n_comparisons, 1.0)  # Bonferroni
            if adj_p < 0.05:
                print(f"    {p1} vs {p2}: U={u_stat:.0f}, p_adj={adj_p:.4f} {'*' if adj_p < 0.05 else ''}")

print(f"\n\nMétricas con diferencias significativas: {len(significant_metrics)}")

## 7. Temperatura de muñeca por fase del ciclo

La temperatura basal sube en la fase lútea por la progesterona. Veamos si esto se refleja en la temperatura de muñeca durante el sueño.

In [ ]:
temp_data = cycle_sleep.dropna(subset=["temp_c"])

if len(temp_data) > 20:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot by phase
    ax = axes[0]
    data = [temp_data[temp_data["phase"] == p]["temp_c"].dropna() for p in phase_order]
    bp = ax.boxplot(data, tick_labels=phase_order, patch_artist=True, widths=0.6)
    for patch, phase in zip(bp["boxes"], phase_order):
        patch.set_facecolor(colors[phase])
        patch.set_alpha(0.6)
    ax.set_title("Temperatura de muñeca por fase", fontsize=13, fontweight="bold")
    ax.set_ylabel("°C")
    
    # Mean temp over cycle day
    ax = axes[1]
    # Normalize cycle day to 0-1 scale
    temp_data = temp_data.copy()
    temp_data["cycle_day_norm"] = temp_data["cycle_day"] / temp_data["cycle_length"]
    temp_data_sorted = temp_data.sort_values("cycle_day_norm")
    # Rolling average
    ax.scatter(temp_data_sorted["cycle_day_norm"], temp_data_sorted["temp_c"], alpha=0.3, s=20)
    # Bin and average
    temp_data["bin"] = pd.cut(temp_data["cycle_day_norm"], bins=20)
    binned = temp_data.groupby("bin", observed=True)["temp_c"].mean()
    bin_centers = [interval.mid for interval in binned.index]
    ax.plot(bin_centers, binned.values, "r-o", linewidth=2, markersize=5, label="Media por bin")
    ax.axvspan(0, 0.18, alpha=0.1, color="red", label="Menstrual")
    ax.axvspan(0.46, 0.57, alpha=0.1, color="green", label="Ovulatoria")
    ax.set_title("Temperatura a lo largo del ciclo (normalizado)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Posición en el ciclo (0=inicio, 1=fin)")
    ax.set_ylabel("°C")
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Statistical test
    groups = [temp_data[temp_data["phase"] == p]["temp_c"].dropna() for p in phase_order]
    stat, p_val = stats.kruskal(*[g for g in groups if len(g) > 0])
    print(f"Kruskal-Wallis temperatura por fase: H={stat:.3f}, p={p_val:.6f}")
else:
    print(f"Insuficientes datos de temperatura con fase asignada ({len(temp_data)} registros)")

## 8. Evolución del sueño a lo largo del ciclo

Visualizamos cómo cambian las métricas de sueño a medida que avanza el ciclo (día 1 → último día).

In [ ]:
# Normalize cycle day position (0 to 1)
cycle_sleep_norm = cycle_sleep.copy()
cycle_sleep_norm["cycle_pos"] = cycle_sleep_norm["cycle_day"] / cycle_sleep_norm["cycle_length"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_metrics = ["total_sleep_min", "pct_rem", "pct_deep", "efficiency"]

for ax, metric in zip(axes.flat, plot_metrics):
    # Bin into 20 equal segments of the cycle
    temp = cycle_sleep_norm[["cycle_pos", metric]].dropna()
    temp["bin"] = pd.cut(temp["cycle_pos"], bins=20)
    binned = temp.groupby("bin", observed=True)[metric].agg(["mean", "sem"])
    bin_centers = [interval.mid for interval in binned.index]
    
    ax.fill_between(bin_centers, 
                    binned["mean"] - 1.96 * binned["sem"],
                    binned["mean"] + 1.96 * binned["sem"],
                    alpha=0.2, color="steelblue")
    ax.plot(bin_centers, binned["mean"], "o-", color="steelblue", linewidth=2, markersize=4)
    
    # Phase boundaries
    ax.axvspan(0, 0.18, alpha=0.08, color="red")
    ax.axvspan(0.46, 0.57, alpha=0.08, color="green")
    ax.axvspan(0.57, 1.0, alpha=0.08, color="orange")
    
    ax.set_title(metric_labels[metric], fontsize=13, fontweight="bold")
    ax.set_xlabel("Posición en el ciclo")

fig.suptitle("Evolución del sueño a lo largo del ciclo menstrual", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 9. Correlación entre métricas

In [ ]:
corr_cols = ["total_sleep_min", "pct_rem", "pct_deep", "pct_core", "efficiency", 
             "n_awakenings", "temp_c", "disturbances", "cycle_day"]
corr_labels = {
    "total_sleep_min": "Duración",
    "pct_rem": "% REM",
    "pct_deep": "% Deep",
    "pct_core": "% Core",
    "efficiency": "Eficiencia",
    "n_awakenings": "Despertares",
    "temp_c": "Temp. muñeca",
    "disturbances": "Pert. respiratorias",
    "cycle_day": "Día del ciclo"
}

corr_data = cycle_sleep[corr_cols].dropna()
corr_matrix = corr_data.corr(method="spearman")
corr_matrix.index = [corr_labels.get(c, c) for c in corr_matrix.index]
corr_matrix.columns = [corr_labels.get(c, c) for c in corr_matrix.columns]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlaciones de Spearman entre métricas de sueño y ciclo", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Efecto premenstrual: Late Luteal vs Early Luteal

La fase lútea tardía (últimos 5 días antes del periodo) es cuando los síntomas premenstruales son más intensos. Separamos la fase lútea en dos para ver si el sueño empeora justo antes del periodo.

In [ ]:
# Split luteal phase into early vs late (last 5 days before next period)
luteal = cycle_sleep[cycle_sleep["phase"] == "Lútea"].copy()
luteal["days_to_next_period"] = luteal["cycle_length"] - luteal["cycle_day"]
label_pre = "Premenstrual\n(<=5d)"
label_early = "Lútea temprana\n(>5d)"
luteal["luteal_sub"] = np.where(luteal["days_to_next_period"] <= 5, label_pre, label_early)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
premenstrual_metrics = ["total_sleep_min", "pct_rem", "n_awakenings", "efficiency"]
premenstrual_labels = ["Duración (min)", "% REM", "Despertares", "Eficiencia (%)"]
sub_colors = {label_pre: "#e74c3c", label_early: "#f39c12"}
sub_order = [label_early, label_pre]

for ax, metric, label in zip(axes, premenstrual_metrics, premenstrual_labels):
    data = [luteal[luteal["luteal_sub"] == s][metric].dropna() for s in sub_order]
    bp = ax.boxplot(data, tick_labels=sub_order, patch_artist=True, widths=0.5)
    for patch, s in zip(bp["boxes"], sub_order):
        patch.set_facecolor(sub_colors[s])
        patch.set_alpha(0.6)
    ax.set_title(label, fontsize=13, fontweight="bold")
    
    u, p = stats.mannwhitneyu(data[0], data[1], alternative="two-sided")
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.text(0.5, 0.02, f"p={p:.4f} {sig}", transform=ax.transAxes, ha="center", fontsize=10, 
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray"))

fig.suptitle("Efecto premenstrual: últimos 5 días antes del periodo vs resto de fase lútea",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

n_early = len(luteal[luteal["luteal_sub"] == label_early])
n_pre = len(luteal[luteal["luteal_sub"] == label_pre])
print(f"Noches lútea temprana: {n_early}")
print(f"Noches premenstruales: {n_pre}")
for metric, mlabel in zip(premenstrual_metrics, premenstrual_labels):
    early = luteal[luteal["days_to_next_period"] > 5][metric].dropna()
    late = luteal[luteal["days_to_next_period"] <= 5][metric].dropna()
    u, p = stats.mannwhitneyu(early, late, alternative="two-sided")
    print(f"  {mlabel}: early={early.mean():.1f} vs pre={late.mean():.1f} (p={p:.4f})")

## 11. HRV (Heart Rate Variability) por fase del ciclo

La HRV refleja el tono del sistema nervioso autónomo. Mayor HRV = mejor recuperación. La progesterona en fase lútea tiende a reducir la HRV al activar el sistema simpático.

In [ ]:
hrv_data = cycle_sleep.dropna(subset=["hrv_ms"])
print(f"Noches con HRV y fase asignada: {len(hrv_data)}")

if len(hrv_data) > 50:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Box plot by phase
    ax = axes[0]
    data = [hrv_data[hrv_data["phase"] == p]["hrv_ms"].dropna() for p in phase_order]
    bp = ax.boxplot(data, tick_labels=phase_order, patch_artist=True, widths=0.6)
    for patch, phase in zip(bp["boxes"], phase_order):
        patch.set_facecolor(colors[phase])
        patch.set_alpha(0.6)
    ax.set_title("HRV (SDNN) por fase", fontsize=13, fontweight="bold")
    ax.set_ylabel("ms")
    
    # HRV over normalized cycle position
    ax = axes[1]
    hrv_norm = hrv_data.copy()
    hrv_norm["cycle_pos"] = hrv_norm["cycle_day"] / hrv_norm["cycle_length"]
    hrv_norm["bin"] = pd.cut(hrv_norm["cycle_pos"], bins=20)
    binned = hrv_norm.groupby("bin", observed=True)["hrv_ms"].agg(["mean", "sem"])
    bin_centers = [interval.mid for interval in binned.index]
    ax.fill_between(bin_centers, binned["mean"] - 1.96*binned["sem"], 
                    binned["mean"] + 1.96*binned["sem"], alpha=0.2, color="steelblue")
    ax.plot(bin_centers, binned["mean"], "o-", color="steelblue", linewidth=2, markersize=4)
    ax.axvspan(0, 0.18, alpha=0.08, color="red")
    ax.axvspan(0.57, 1.0, alpha=0.08, color="orange")
    ax.set_title("HRV a lo largo del ciclo", fontsize=13, fontweight="bold")
    ax.set_xlabel("Posición en el ciclo")
    ax.set_ylabel("ms")
    
    # HRV vs temperature scatter
    ax = axes[2]
    both = cycle_sleep.dropna(subset=["hrv_ms", "temp_c"])
    if len(both) > 20:
        ax.scatter(both["temp_c"], both["hrv_ms"], alpha=0.3, s=15, c=[colors[p] for p in both["phase"]])
        rho, p = stats.spearmanr(both["temp_c"], both["hrv_ms"])
        ax.set_title(f"HRV vs Temperatura (rho={rho:.3f}, p={p:.4f})", fontsize=12, fontweight="bold")
        ax.set_xlabel("Temp muñeca (°C)")
        ax.set_ylabel("HRV (ms)")
    
    plt.tight_layout()
    plt.show()
    
    # Kruskal-Wallis
    groups = [hrv_data[hrv_data["phase"] == p]["hrv_ms"].dropna() for p in phase_order]
    stat, p_val = stats.kruskal(*[g for g in groups if len(g) > 5])
    print(f"\nKruskal-Wallis HRV por fase: H={stat:.3f}, p={p_val:.6f}")
    for p in phase_order:
        vals = hrv_data[hrv_data["phase"] == p]["hrv_ms"]
        print(f"  {p}: {vals.mean():.1f} ± {vals.std():.1f} ms (n={len(vals)})")
else:
    print("Insuficientes datos de HRV")

## 12. Frecuencia cardíaca en reposo por fase del ciclo

La frecuencia cardíaca en reposo (RHR) sube ~2-5 bpm en fase lútea por el efecto termogénico y simpático de la progesterona.

In [ ]:
rhr_data = cycle_sleep.dropna(subset=["resting_hr_bpm"])
print(f"Noches con resting HR y fase asignada: {len(rhr_data)}")

if len(rhr_data) > 50:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot by phase
    ax = axes[0]
    data = [rhr_data[rhr_data["phase"] == p]["resting_hr_bpm"].dropna() for p in phase_order]
    bp = ax.boxplot(data, tick_labels=phase_order, patch_artist=True, widths=0.6)
    for patch, phase in zip(bp["boxes"], phase_order):
        patch.set_facecolor(colors[phase])
        patch.set_alpha(0.6)
    ax.set_title("Resting HR por fase", fontsize=13, fontweight="bold")
    ax.set_ylabel("bpm")
    
    # RHR over normalized cycle position
    ax = axes[1]
    rhr_norm = rhr_data.copy()
    rhr_norm["cycle_pos"] = rhr_norm["cycle_day"] / rhr_norm["cycle_length"]
    rhr_norm["bin"] = pd.cut(rhr_norm["cycle_pos"], bins=20)
    binned = rhr_norm.groupby("bin", observed=True)["resting_hr_bpm"].agg(["mean", "sem"])
    bin_centers = [interval.mid for interval in binned.index]
    ax.fill_between(bin_centers, binned["mean"] - 1.96*binned["sem"],
                    binned["mean"] + 1.96*binned["sem"], alpha=0.2, color="indianred")
    ax.plot(bin_centers, binned["mean"], "o-", color="indianred", linewidth=2, markersize=4)
    ax.axvspan(0, 0.18, alpha=0.08, color="red")
    ax.axvspan(0.57, 1.0, alpha=0.08, color="orange")
    ax.set_title("Resting HR a lo largo del ciclo", fontsize=13, fontweight="bold")
    ax.set_xlabel("Posición en el ciclo")
    ax.set_ylabel("bpm")
    
    plt.tight_layout()
    plt.show()
    
    groups = [rhr_data[rhr_data["phase"] == p]["resting_hr_bpm"].dropna() for p in phase_order]
    stat, p_val = stats.kruskal(*[g for g in groups if len(g) > 5])
    print(f"\nKruskal-Wallis resting HR por fase: H={stat:.3f}, p={p_val:.6f}")
    for p in phase_order:
        vals = rhr_data[rhr_data["phase"] == p]["resting_hr_bpm"]
        print(f"  {p}: {vals.mean():.1f} ± {vals.std():.1f} bpm (n={len(vals)})")
else:
    print("Insuficientes datos de resting HR")

## 13. Cronobiología: hora de acostarse y despertarse por fase

¿Cambia el cronotipo con la fase del ciclo? La progesterona tiene efecto sedante, lo que podría adelantar la hora de dormir en fase lútea.

In [ ]:
# Extract bedtime and wake time as hours (decimal)
timing = cycle_sleep[["night_date", "phase", "sleep_start", "sleep_end", "cycle_day", "cycle_length"]].copy()
timing["bedtime_hour"] = timing["sleep_start"].dt.hour + timing["sleep_start"].dt.minute / 60
# Adjust: if bedtime is before 18:00, add 24 (next day assignment issue)
timing.loc[timing["bedtime_hour"] < 18, "bedtime_hour"] += 24
timing["waketime_hour"] = timing["sleep_end"].dt.hour + timing["sleep_end"].dt.minute / 60

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bedtime by phase
ax = axes[0]
data = [timing[timing["phase"] == p]["bedtime_hour"].dropna() for p in phase_order]
bp = ax.boxplot(data, tick_labels=phase_order, patch_artist=True, widths=0.6)
for patch, phase in zip(bp["boxes"], phase_order):
    patch.set_facecolor(colors[phase])
    patch.set_alpha(0.6)
ax.set_title("Hora de acostarse por fase", fontsize=13, fontweight="bold")
ax.set_ylabel("Hora")
ax.set_yticks([22, 23, 24, 25, 26, 27])
ax.set_yticklabels(["22:00", "23:00", "0:00", "1:00", "2:00", "3:00"])

# Wake time by phase
ax = axes[1]
data = [timing[timing["phase"] == p]["waketime_hour"].dropna() for p in phase_order]
bp = ax.boxplot(data, tick_labels=phase_order, patch_artist=True, widths=0.6)
for patch, phase in zip(bp["boxes"], phase_order):
    patch.set_facecolor(colors[phase])
    patch.set_alpha(0.6)
ax.set_title("Hora de despertarse por fase", fontsize=13, fontweight="bold")
ax.set_ylabel("Hora")

plt.tight_layout()
plt.show()

# Stats
for label, col in [("Bedtime", "bedtime_hour"), ("Wake time", "waketime_hour")]:
    groups = [timing[timing["phase"] == p][col].dropna() for p in phase_order]
    stat, p_val = stats.kruskal(*groups)
    print(f"{label}: H={stat:.3f}, p={p_val:.6f}")
    for p in phase_order:
        vals = timing[timing["phase"] == p][col]
        if col == "bedtime_hour":
            h, m = divmod(int(vals.mean() % 24 * 60 + 0.5) if vals.mean() < 24 else int((vals.mean()-24)*60+0.5), 60)
            if vals.mean() >= 24: h = h  # already correct for display
            print(f"  {p}: {vals.mean():.2f}h (~{int(vals.mean()%24):02d}:{int(vals.mean()%1*60):02d})")
        else:
            print(f"  {p}: {vals.mean():.2f}h (~{int(vals.mean()):02d}:{int(vals.mean()%1*60):02d})")

## 14. Tendencia temporal: evolución del sueño en 6 años

¿Ha cambiado la calidad del sueño con el tiempo? Separamos el efecto del ciclo del efecto temporal.

In [ ]:
# Monthly aggregation of sleep metrics
monthly = cycle_sleep.copy()
monthly["month"] = monthly["night_date"].dt.to_period("M")

monthly_agg = monthly.groupby("month").agg(
    sleep_min=("total_sleep_min", "mean"),
    pct_rem=("pct_rem", "mean"),
    pct_deep=("pct_deep", "mean"),
    efficiency=("efficiency", "mean"),
    n_awakenings=("n_awakenings", "mean"),
    n_nights=("night_date", "count")
).reset_index()
monthly_agg["month_dt"] = monthly_agg["month"].dt.to_timestamp()
monthly_agg = monthly_agg[monthly_agg["n_nights"] >= 10]  # Only months with enough data

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
trend_metrics = [("sleep_min", "Duración media (min)"), ("pct_rem", "% REM medio"),
                 ("pct_deep", "% Deep medio"), ("n_awakenings", "Despertares medios")]

for ax, (col, label) in zip(axes.flat, trend_metrics):
    ax.plot(monthly_agg["month_dt"], monthly_agg[col], "o-", markersize=3, alpha=0.7, color="steelblue")
    # Trend line
    x_num = (monthly_agg["month_dt"] - monthly_agg["month_dt"].min()).dt.days.values
    valid = ~np.isnan(monthly_agg[col].values)
    if valid.sum() > 5:
        z = np.polyfit(x_num[valid], monthly_agg[col].values[valid], 1)
        ax.plot(monthly_agg["month_dt"], np.polyval(z, x_num), "--", color="red", alpha=0.7,
                label=f"Tendencia: {z[0]*365:.2f}/año")
    ax.set_title(label, fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Evolución temporal del sueño (medias mensuales)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Cycle length trend
valid_cycles = periods.dropna(subset=["cycle_length"])
valid_cycles = valid_cycles[(valid_cycles["cycle_length"] >= 21) & (valid_cycles["cycle_length"] <= 45)]
if len(valid_cycles) > 10:
    x_num = (valid_cycles["start"] - valid_cycles["start"].min()).dt.days.values
    z = np.polyfit(x_num, valid_cycles["cycle_length"].values, 1)
    print(f"Tendencia duración del ciclo: {z[0]*365:.2f} días/año")
    print(f"  Primer año: ~{valid_cycles['cycle_length'].head(12).mean():.1f} días")
    print(f"  Último año: ~{valid_cycles['cycle_length'].tail(12).mean():.1f} días")

## 15. Predicción de fase del ciclo a partir de biomarcadores

¿Podemos predecir la fase del ciclo usando solo temperatura, HRV y frecuencia cardíaca? Un clasificador simple que demuestre que estos biomarcadores contienen información sobre la fase hormonal.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# Prepare features
pred_cols = ["temp_c", "hrv_ms", "resting_hr_bpm"]
pred_data = cycle_sleep.dropna(subset=pred_cols + ["phase"]).copy()

if len(pred_data) > 100:
    X = pred_data[pred_cols].values
    y = pred_data["phase"].values
    
    # Binary: Luteal vs Non-Luteal (easier task, clinically relevant)
    y_binary = np.where(y == "Lútea", "Lútea", "No-Lútea")
    
    clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X, y_binary, cv=cv, scoring="f1_macro")
    
    print(f"Clasificación Lútea vs No-Lútea (5-fold CV)")
    print(f"  F1-macro: {scores.mean():.3f} ± {scores.std():.3f}")
    print(f"  Scores: {[f'{s:.3f}' for s in scores]}")
    
    # Train on all data for feature importance
    clf.fit(X, y_binary)
    print(f"\nImportancia de features:")
    for name, imp in sorted(zip(pred_cols, clf.feature_importances_), key=lambda x: -x[1]):
        print(f"  {name}: {imp:.3f}")
    
    # Full report
    from sklearn.model_selection import cross_val_predict
    y_pred = cross_val_predict(clf, X, y_binary, cv=cv)
    print(f"\nClassification Report:")
    print(classification_report(y_binary, y_pred))
    
    # 4-class
    print("\n--- Clasificación 4 fases ---")
    scores_4 = cross_val_score(clf, X, y, cv=cv, scoring="f1_macro")
    print(f"  F1-macro: {scores_4.mean():.3f} ± {scores_4.std():.3f}")
    
    # Confusion matrix
    y_pred_4 = cross_val_predict(RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced"), X, y, cv=cv)
    cm = confusion_matrix(y, y_pred_4, labels=phase_order)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=phase_order, yticklabels=phase_order, ax=ax)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_title("Matriz de confusión: predicción de fase del ciclo\n(temp + HRV + resting HR)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print(f"Insuficientes datos con todos los biomarcadores ({len(pred_data)} registros)")
    print("Se necesitan noches con temperatura, HRV y resting HR simultáneamente")

## 16. Resumen de hallazgos

In [ ]:
# Complete summary with all metrics
all_bio_metrics = ["total_sleep_min", "pct_rem", "pct_deep", "efficiency", "n_awakenings", 
                   "temp_c", "hrv_ms", "resting_hr_bpm", "disturbances"]
all_bio_labels = {
    "total_sleep_min": "Duración (min)", "pct_rem": "% REM", "pct_deep": "% Deep",
    "efficiency": "Eficiencia (%)", "n_awakenings": "Despertares",
    "temp_c": "Temp muñeca (°C)", "hrv_ms": "HRV (ms)", 
    "resting_hr_bpm": "Resting HR (bpm)", "disturbances": "Pert. respiratorias"
}

summary_all = cycle_sleep.groupby("phase")[all_bio_metrics].mean().round(2)
summary_all = summary_all.reindex(phase_order)
summary_all.columns = [all_bio_labels.get(c, c) for c in summary_all.columns]

print("RESUMEN COMPLETO: Media de todas las métricas por fase del ciclo")
print("=" * 80)
display(summary_all)

# Statistical significance summary
print("\nSignificancia estadística (Kruskal-Wallis):")
print("-" * 60)
for metric in all_bio_metrics:
    groups = [cycle_sleep[cycle_sleep["phase"] == p][metric].dropna() for p in phase_order]
    valid = [g for g in groups if len(g) > 5]
    if len(valid) >= 2:
        stat, p_val = stats.kruskal(*valid)
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"  {all_bio_labels.get(metric, metric):25s}: p={p_val:.6f} [{sig}]")

In [ ]:
# Final radar chart with all normalized metrics
radar_metrics = ["total_sleep_min", "pct_rem", "pct_deep", "efficiency"]
radar_labels = ["Duración", "% REM", "% Deep", "Eficiencia"]

phase_means = cycle_sleep.groupby("phase")[radar_metrics].mean()
phase_means = phase_means.reindex(phase_order)
normalized = (phase_means - phase_means.min()) / (phase_means.max() - phase_means.min())

angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for phase in phase_order:
    values = normalized.loc[phase].tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, label=phase, color=colors[phase])
    ax.fill(angles, values, alpha=0.1, color=colors[phase])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=12)
ax.set_title("Perfil de sueño por fase del ciclo\n(valores normalizados)", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()